# 旅行商问题 (TSP)

**类别：** 路径规划

来源: [https://www.hexaly.com/templates/traveling-salesman-problem-tsp](https://www.hexaly.com/templates/traveling-salesman-problem-tsp)


## 问题

**旅行商问题 (TSP)** 定义如下：给定 n 个城市以及每对城市之间的距离，寻找一条总长度最短的环游路径，使其恰好访问每个城市一次。从城市 i 到城市 j 的距离与从城市 j 到城市 i 的距离可能不同。

	

### 学到的建模原则

- 添加 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模城市的排列
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算行驶距离
- 获取 [list 变量的取值](https://www.hexaly.com/docs/last/features/solution.html#values-of-collection-and-array-variables-and-expressions)


## 数据

所提供的旅行商问题 (TSP) 实例来自 [TSPLib](http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/) 非对称 TSP 数据库，采用 TSPLib 显式格式。城市数量在关键字 “DIMENSION:” 之后定义，完整的距离矩阵在关键字 “EDGE_WEIGHT_SECTION” 之后给出。


## 模型

该 Hexaly 模型基于一个 list 决策变量。list 变量的第 i 个元素对应路径中第 i 个访问城市的索引。首先，我们约束该 list 包含所有城市的索引，以确保所有城市都被访问。根据该 list，我们可以直接得到 list 中每对相邻城市之间的距离，再加上闭合边（从最后一个城市回到第一个城市）的距离。注意，这里我们使用了二维 [‘at’ 运算符](https://www.hexaly.com/docs/last/mathematicaloperators/atoperator.html) z <- A[x][y]，它将 z 定义为矩阵 A 中元素 (x, y)，其中 x 和 y 是整数表达式。该运算符允许在三个变量 x、y、z 之间定义几乎任意形式的非线性关系。我们还使用了 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)，以便在整个城市范围上应用 **sum（求和）** 运算符。


## 结果

在 TSPLib 研究基准上，对于最多 **10,000 个城市**的实例，Hexaly Optimizer 能够在 **1 分钟**运行时间内，使旅行商问题 (TSP) 的**平均最优性差距达到 0.3%**。我们的 [旅行商 (TSP) 基准测试页面](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-traveling-salesman-problem-tsp)展示了 Hexaly Optimizer 在这一具有挑战性的组合优化问题上如何超越 Gurobi 11.0 等传统通用优化求解器。

[查看该基准测试](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-traveling-salesman-problem-tsp)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python tsp.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)


def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read instance data
    #
    file_it = iter(read_elem(sys.argv[1]))

    # The input files follow the TSPLib "explicit" format
    for pch in file_it:
        if pch == "DIMENSION:":
            nb_cities = int(next(file_it))
        if pch == "EDGE_WEIGHT_SECTION":
            break

    # Distance from i to j
    dist_matrix_data = [[int(next(file_it)) for i in range(nb_cities)]
                        for j in range(nb_cities)]

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # A list variable: cities[i] is the index of the ith city in the tour
    cities = model.list(nb_cities)

    # All cities must be visited
    model.constraint(model.count(cities) == nb_cities)

    # Create an Hexaly array for the distance matrix in order to be able
    # to access it with "at" operators
    dist_matrix = model.array(dist_matrix_data)

    # Minimize the total distance
    dist_lambda = model.lambda_function(lambda i:
                                        model.at(dist_matrix, cities[i - 1], cities[i]))
    obj = model.sum(model.range(1, nb_cities), dist_lambda) \
        + model.at(dist_matrix, cities[nb_cities - 1], cities[0])
    model.minimize(obj)

    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 5

    optimizer.solve()

    #
    # Write the solution in a file
    #
    if len(sys.argv) >= 3:
        # Write the solution in a file
        with open(sys.argv[2], 'w') as f:
            f.write("%d\n" % obj.value)
            for c in cities.value:
                f.write("%d " % c)
            f.write("\n")
